# Random Forest Model

Random Forest is an ensemble learning algorithm that builds multiple decision trees
and combines their predictions to improve accuracy and reduce overfitting.

In this notebook:
- We train the Random Forest model
- Apply threshold tuning
- Evaluate model performance

In [9]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score,
                             confusion_matrix, roc_auc_score)

In [18]:
X_train = joblib.load("../models/X_train.pkl")
X_test  = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test  = joblib.load("../models/y_test.pkl")
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Class dist:", np.bincount(y_train))

Train: (565424, 51) | Test: (141357, 51)
Class dist: [557378   8046]


In [19]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight='balanced_subsample',  # RF-specific: balances each tree's bootstrap sample
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train, y_train)
joblib.dump(rf, "../models/random_forest_model.pkl")
print("Random Forest saved.")

Random Forest saved.


We use a custom threshold (0.6) instead of the default 0.5
to balance Precision and Recall.

In [21]:
y_proba = rf.predict_proba(X_test)[:, 1]

print(f"{'Thresh':>8} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-" * 45)
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.20, 0.61, 0.05):
    yp = (y_proba >= t).astype(int)
    pr = precision_score(y_test, yp, zero_division=0)
    re = recall_score(y_test, yp, zero_division=0)
    f1 = f1_score(y_test, yp, zero_division=0)
    print(f"{t:>8.2f} {accuracy_score(y_test,yp):>8.3f} {pr:>8.3f} {re:>8.3f} {f1:>8.3f}")
    if f1 > best_f1 and pr >= 0.30 and re >= 0.30:
        best_f1, best_thresh = f1, t

print(f"\nBest threshold: {best_thresh:.2f}  F1={best_f1:.3f}")
joblib.dump(best_thresh, "../models/rf_threshold.pkl")

  Thresh      Acc     Prec      Rec       F1
---------------------------------------------
    0.20    0.338    0.021    0.998    0.041
    0.25    0.522    0.029    0.993    0.056
    0.30    0.654    0.038    0.970    0.074
    0.35    0.747    0.051    0.951    0.097
    0.40    0.804    0.062    0.910    0.117
    0.45    0.833    0.068    0.843    0.125
    0.50    0.982    0.413    0.620    0.496
    0.55    0.988    0.567    0.517    0.541
    0.60    0.989    0.673    0.427    0.522

Best threshold: 0.55  F1=0.541


['../models/rf_threshold.pkl']

In [22]:
THRESHOLD = best_thresh
y_pred = (y_proba >= THRESHOLD).astype(int)
print(f"===== Random Forest Results (threshold={THRESHOLD:.2f}) =====")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1 Score : {f1_score(y_test, y_pred, zero_division=0):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba):.3f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

===== Random Forest Results (threshold=0.55) =====
Accuracy : 0.988
Precision: 0.567
Recall   : 0.517
F1 Score : 0.541
ROC-AUC  : 0.938
Confusion Matrix:
[[138552    794]
 [   971   1040]]
